In [ ]:
import os
import re
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.stats import linregress
from pathlib import Path
from abc import ABCMeta, abstractmethod
from time import time
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression

In [ ]:
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.util import Logger, Util
from src.feature import *

In [ ]:
import importlib
import src.feature
importlib.reload(src.feature)
from src.feature import *

In [ ]:
pd.set_option("display.max_columns",500)
pd.set_option("display.max_rows", 500)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 処理実行

In [ ]:
def run_blocks(feature_blocks):
    print('start run blocks...')
    with Timer(prefix='run test'):
        for block in feature_blocks:
            with Timer(prefix='\t- {}'.format(str(block))):
                feature = block.create_feature()

In [ ]:
feature_blocks = [
    Key(use_cache=False, save_cache=True, logger=None),
	Target(use_cache=False, save_cache=True, logger=None),
    CategoryFeature(use_cache=False, save_cache=True, logger=None),
	CareerFeature(use_cache=False, save_cache=True, logger=None),
	UdemyActivityFeature(use_cache=False, save_cache=True, logger=None),
    UdemyTimeseriesFeature(use_cache=False, save_cache=True, logger=None),
    UdemyTitleEmbedding(use_cache=False, save_cache=True, logger=None),
	UdemyIDEmbedding(use_cache=False, save_cache=True, logger=None),
    UdemyCategorySimilarityFeature(use_cache=True, save_cache=True, logger=None),
    UdemyTitleSimilarityFeature(use_cache=False, save_cache=True, logger=None),
	DxFeature(use_cache=False, save_cache=True, logger=None),
	HrFeature(use_cache=False, save_cache=True, logger=None),
	OvertimeWorkByMonthFeature(use_cache=False, save_cache=True, logger=None),
    OvertimeWorkByMonthTimeseriesFeature(use_cache=True, save_cache=True, logger=None),
	PositionHistoryFeature(use_cache=False, save_cache=True, logger=None),
]

In [ ]:
run_blocks(feature_blocks)

In [ ]:
list_ = [
    'Key',
    'Target',
    'CategoryFeature',
    'CareerFeature',
    'UdemyActivityFeature',
    'UdemyTimeseriesFeature',
    'UdemyTitleEmbedding',
    'UdemyCategorySimilarityFeature',
    'UdemyTitleSimilarityFeature',
    'UdemyIDEmbedding',
    'DxFeature',
    'HrFeature',
    'OvertimeWorkByMonthFeature',
    'OvertimeWorkByMonthTimeseriesFeature',
    'PositionHistoryFeature',
]
dict_shape = {}
for feature_name in list_:
    dict_shape[feature_name] = Util.load_feature(feature_name).shape
pd.DataFrame(dict_shape, index=['n_rows', 'n_cols']).T

In [ ]:
# データ読み込み
df_train = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_train.pkl"))
df_udemy = pd.read_pickle(os.path.join(DIR_INTERIM, "df_prep_udemy_activity.pkl"))

# nan除外
df_udemy = df_udemy[df_udemy["コースタイトル"].notnull()].copy()

# ユニークな講座カテゴリとtrainカテゴリを抽出
unique_udemy_cats = df_udemy["コースタイトル"].unique().tolist()
unique_train_cats = df_train["category"].unique().tolist()

# 埋め込みを取得
model_name = "hotchpotch/static-embedding-japanese"
model = SentenceTransformer(model_name, device="cpu")
emb_udemy = model.encode(unique_udemy_cats, show_progress_bar=True)
emb_train = model.encode(unique_train_cats, show_progress_bar=True)

# 類似度行列 (trainカテゴリ x udemyカテゴリ)
sim_matrix = cosine_similarity(emb_train, emb_udemy)
df_sim = pd.DataFrame(sim_matrix, index=unique_train_cats, columns=unique_udemy_cats)

# 社員ごとの受講履歴を重複ありで取得
# df_user_course = df_udemy[["社員番号", "コースタイトル"]].copy()
df_user_course = df_udemy[["社員番号", "コースタイトル"]].drop_duplicates().copy()

df_category_sim_feature = df_train[["社員番号", "category"]].drop_duplicates().copy()

# 類似度スコアの統計量（平均・最大など）を算出
sim_mean_list = []
sim_max_list = []
sim_min_list = []
for _, row in df_category_sim_feature.iterrows():
    emp_id = row["社員番号"]
    train_cat = row["category"]
    # その社員が受講したコースタイトル
    learned_cats = df_user_course[df_user_course["社員番号"] == emp_id]["コースタイトル"].tolist()
    # 公募カテゴリとの類似度を取得
    similarities = [df_sim.loc[train_cat, cat] for cat in learned_cats]
    # 類似度スコアの統計量（平均・最大など）を算出
    if similarities:
        sim_mean_list.append(np.mean(similarities))
        sim_max_list.append(np.max(similarities))
        sim_min_list.append(np.min(similarities))
    else:
        sim_mean_list.append(np.nan)
        sim_max_list.append(np.nan)
        sim_min_list.append(np.nan)

# 結果をDataFrameに追加
df_category_sim_feature["ua_コースタイトル_sim_mean"] = sim_mean_list
# df_category_sim_feature["ua_コースカテゴリ_sim_max"] = sim_max_list
# df_category_sim_feature["ua_コースカテゴリ_sim_min"] = sim_min_list

In [ ]:
df_sim

In [ ]:
df_category_sim_feature